In [13]:
!pip install datasets
!pip install evaluate

In [14]:
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader
from dataclasses import dataclass
from datasets import load_dataset, concatenate_datasets, ClassLabel
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification, AutoTokenizer
import torch

import evaluate
import numpy as np

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import warnings
warnings.filterwarnings('ignore')

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

#### Constants, that will be used for training, loading, process, tokenizing data.

In [15]:
DISTIL_BERT = "distilbert-base-uncased"
ROBERTA_BASE = "roberta-base"

DATASETS_LINKS: list[str] = [
    'AdamCodd/emotion-balanced',
    'dair-ai/emotion',
    # 'philschmid/emotion',
    'SetFit/emotion',
    'mteb/emotion'
]

#### Data classes that is going to served as an interfaces for the classes

In [16]:
@dataclass
class DataLoaderSettings:
    dataset_link: str
    keys: list[str]
    text_col: str
    label_col: str

@dataclass
class DatasetSettings:
    label_col: str
    text_col: str
    tokenizer_link: str

@dataclass
class TrainingInformation:
    pretrained_model: str

#### Data Loader, will load the data, process (tokenize, remove stop words and other important things), convert labels to class label.

In [17]:
class DataLoader:
    '''

    '''
    def __init__(self, loader_settings: DataLoaderSettings):
        self.stop_words = stopwords.words('english')
        self.settings = loader_settings
        self.loaded = self.load_dataset()
        self.processed = self.process()
        self.convert_labels_to_classlabel()

        print(f'Loaded: {self.loaded}')
        print(f'Processed: {self.processed}')

    def load_dataset(self) -> bool:
        try:
            dataset = load_dataset(self.settings.dataset_link, trust_remote_code=True)
            merged_dataset = None

            if not isinstance(dataset, dict):
                self.dataset = dataset
                return True

            for key in self.settings.keys:
                if key not in dataset:
                    continue

                dataset_partition = dataset[key]

                if merged_dataset == None:
                    merged_dataset = dataset_partition
                else:
                    merged_dataset = concatenate_datasets([merged_dataset, dataset_partition])

            self.dataset = merged_dataset
            return True
        except Exception as e:
            print(f'Something went wrong when trying to load dataset from link {self.settings.dataset_link}')
            print(f'Got error {e}')
            return False

    def convert_labels_to_classlabel(self):
        unique_labels = list(set(self.dataset[self.settings.label_col]))
        class_label_feature = ClassLabel(num_classes=len(unique_labels), names=[str(label) for label in unique_labels])

        self.dataset = self.dataset.map(lambda example: {self.settings.label_col: class_label_feature.str2int(str(example[self.settings.label_col]))})
        self.dataset = self.dataset.cast_column(self.settings.label_col, class_label_feature)
        print(type(self.dataset[0]['label']))

    def process(self) -> bool:
        if not self.loaded:
            raise ValueError("Dataset has not been loaded")

        def process_text(sample) -> str:
            text = sample[self.settings.text_col]
            words = self.tokenize(text)
            words = self.remove_stopwords(words)

            sample[self.settings.text_col] = ' '.join(words)
            return sample

        try:
            self.dataset = self.dataset.map(process_text)
            return True
        except Exception as e:
            print(e)
            return False

    def tokenize(self, text) -> list[str]:
        words = word_tokenize(text)
        return words

    def remove_stopwords(self, words) -> list[str]:
        words = [word for word in words if word not in self.stop_words and word.isalpha()]
        return words

#### Custom dataset that is going to be used for splitting dataset, tokenizing dataset.

In [18]:
class CustomDataset:
    """
    Custom dataset class for tokenizing text data.

    Attributes:
    - dataset: DataFrame loaded from CSV
    - tokenizer: Tokenizer for text processing
    - max_length: Maximum token length
    """

    def __init__(self, dataset_settings: DatasetSettings, data_loader: DataLoader, max_length=512):
        self.tokenizer = AutoTokenizer.from_pretrained(dataset_settings.tokenizer_link)
        self.settings = dataset_settings
        self.data_loader = data_loader
        self.max_length = max_length
        self.label_encoder = LabelEncoder()
        self.splitted = self.split_dataset()
        self.tokenized = self.tokenize_datasets()
        print(f'Splitted: {self.splitted}')
        print(f'Tokenized: {self.tokenized}')

    def count_unique_labels(self) -> int:
        """
        Counts the number of unique labels in the dataset.

        Returns:
        - int: The number of unique labels.
        """
        try:
            label_column = self.settings.label_col

            # Get unique labels from dataset
            unique_labels = set(self.train[label_column])

            print(f"Number of unique labels: {len(unique_labels)}")
            return len(unique_labels)

        except Exception as e:
            print(f"Error counting unique labels: {e}")
            return 0

    def __len__(self):
        return len(self.data)

    def split_dataset(self, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42) -> bool:
        """
        Split the dataset into training, validation, and testing sets.

        Args:
        - train_ratio (float): Proportion of the dataset to use for training.
        - val_ratio (float): Proportion of the dataset to use for validation.
        - test_ratio (float): Proportion of the dataset to use for testing.
        - random_state (int): Seed for reproducibility.

        Returns:
        - bool: True if the split was successful, False otherwise.
        """
        if not hasattr(self, "data_loader"):
            print("Dataset is not loaded.")
            return False

        if not (0 < train_ratio < 1 and 0 < val_ratio < 1 and 0 < test_ratio < 1 and train_ratio + val_ratio + test_ratio == 1):
            print("Invalid split ratios. Ensure they sum to 1.")
            return False

        dataset = self.data_loader.dataset

        try:
            # First, split into train and temp (val + test)
            train_test_split = dataset.train_test_split(test_size=(1 - train_ratio), seed=seed, stratify_by_column=self.settings.label_col)
            train_data = train_test_split["train"]
            temp_data = train_test_split["test"]

            # Compute relative validation split
            val_size = val_ratio / (val_ratio + test_ratio)  # Normalize val/test split
            val_test_split = temp_data.train_test_split(test_size=(1 - val_size), seed=seed, stratify_by_column=self.settings.label_col)

            self.train = train_data
            self.val = val_test_split["train"]
            self.test = val_test_split["test"]

            print(f"Dataset split complete: Train({len(self.train)}), Val({len(self.val)}), Test({len(self.test)})")
            return True

        except Exception as e:
            print(f"Error splitting dataset: {e}")
            return False

    def tokenize_datasets(self):
        """
        Tokenizes the train, validation, and test datasets using the tokenizer.

        This function modifies self.train, self.val, and self.test in-place.
        """
        if not hasattr(self, "train") or self.train is None:
            print("Training dataset is not loaded.")
            return False
        if not hasattr(self, "val") or self.val is None:
            print("Validation dataset is not loaded.")
            return False
        if not hasattr(self, "test") or self.test is None:
            print("Testing dataset is not loaded.")
            return False

        try:
            text_column = self.settings.text_col
            label_column = self.settings.label_col

            # Tokenization function
            def tokenize_function(example):
                encoding = self.tokenizer(
                    example[text_column],
                    padding="max_length",
                    truncation=True,
                    max_length=self.max_length
                )
                encoding["labels"] = [torch.tensor(label, dtype=torch.long) for label in example[label_column]]
                return encoding

            self.train = self.train.map(tokenize_function, batched=True)
            self.val = self.val.map(tokenize_function, batched=True)
            self.test = self.test.map(tokenize_function, batched=True)

            print("Tokenization complete for train, val, and test datasets.")
            return True

        except Exception as e:
            print(f"Error tokenizing datasets: {e}")
            return False

#### Training script and functions with additional evaluation metrics.

In [19]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

In [20]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    precision = precision_metric.compute(predictions=predictions, references=labels, average="macro")
    recall = recall_metric.compute(predictions=predictions, references=labels, average="macro")
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")

    return {
        "accuracy": accuracy["accuracy"],
        "precision": precision["precision"],
        "recall": recall["recall"],
        "f1": f1["f1"]
    }

def train_model(dataset: CustomDataset, training_information: TrainingInformation):
    model = AutoModelForSequenceClassification.from_pretrained(training_information.pretrained_model, num_labels=dataset.count_unique_labels())

    training_args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir="./logs",
        logging_steps=10,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset.train,
        eval_dataset=dataset.test,
        compute_metrics=compute_metrics
    )
    trainer.train()

    evaluation_result = trainer.evaluate(dataset.test)
    print("EVALUATION RESULT")
    print(evaluation_result)

In [21]:
datasets: list[CustomDataset] = []

for dataset_link in DATASETS_LINKS:
    print(dataset_link)
    print('')

    loader_settings = DataLoaderSettings(
        dataset_link=dataset_link,
        keys=['train', 'validation', 'test'],
        label_col='label',
        text_col='text'
    )

    loader = DataLoader(loader_settings=loader_settings)
    print(loader.dataset[0])

    dataset_settings = DatasetSettings(
        tokenizer_link=ROBERTA_BASE,
        label_col='label',
        text_col='text'
    )

    dataset = CustomDataset(
        data_loader=loader,
        dataset_settings=dataset_settings
    )

    datasets.append(dataset)

AdamCodd/emotion-balanced

<class 'int'>
Loaded: True
Processed: True
{'text': 'sick feeling like want opinions please nothing rude imature', 'label': 3}


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True
mteb/emotion

<class 'int'>
Loaded: True
Processed: True
{'text': 'didnt feel humiliated', 'label': 0, 'label_text': 'sadness'}
Dataset split complete: Train(16000), Val(2000), Test(2000)


Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenization complete for train, val, and test datasets.
Splitted: True
Tokenized: True


In [22]:
len(datasets)

4

In [ ]:
for idx in range(len(datasets)):
    print(DATASETS_LINKS[idx])

    training_information = TrainingInformation(
        pretrained_model=ROBERTA_BASE
    )

    train_model(datasets[idx], training_information=training_information)

AdamCodd/emotion-balanced
Number of unique labels: 6


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.344600,0.256553,0.933000,0.935901,0.933000,0.932710
2,0.162500,0.149101,0.957500,0.958548,0.957477,0.957347
3,0.167300,0.167286,0.958000,0.958871,0.957979,0.957851


EVALUATION RESULT
{'eval_loss': 0.1672857403755188, 'eval_accuracy': 0.958, 'eval_precision': 0.9588713892359936, 'eval_recall': 0.9579789370208532, 'eval_f1': 0.9578510033437985, 'eval_runtime': 54.5164, 'eval_samples_per_second': 36.686, 'eval_steps_per_second': 4.586, 'epoch': 3.0}
dair-ai/emotion
Number of unique labels: 6


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.339300,0.289268,0.922500,0.886283,0.912853,0.896809
2,0.030200,0.197172,0.934000,0.918097,0.911091,0.908796
3,0.290500,0.160916,0.939000,0.927177,0.910134,0.916574


EVALUATION RESULT
{'eval_loss': 0.160916268825531, 'eval_accuracy': 0.939, 'eval_precision': 0.9271767451881335, 'eval_recall': 0.91013402974789, 'eval_f1': 0.9165735573211435, 'eval_runtime': 54.6431, 'eval_samples_per_second': 36.601, 'eval_steps_per_second': 4.575, 'epoch': 3.0}
SetFit/emotion
Number of unique labels: 6


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.339300,0.289268,0.922500,0.886283,0.912853,0.896809
